In [1]:
import json
from dotenv import load_dotenv
from openai.types.chat import ChatCompletionMessage, ChatCompletionToolParam

from tool_utils import get_current_time
from tool_runner import run_tool
from memory import add_messages, get_messages, save_tool_response, AIMessage

load_dotenv()

True

In [2]:
import aisuite as ai

client = ai.Client()

In [3]:
def run_llm(messages: list[AIMessage], tools: list[ChatCompletionToolParam], model: str = "openai:gpt-4o") -> ChatCompletionMessage:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools, # <-- Your list of tools with get_current_time
    )

    return response.choices[0].message


def run_agent(user_message: str, tools: list[ChatCompletionToolParam]) -> None:
    max_turns = 0
    add_messages([{"role": "user", "content": user_message}])

    while True:
        history: list[AIMessage] = get_messages()
        if max_turns > 4:
            print(f"Reached max AI Conversation of ${max_turns} turns")
            return history
        
        response: ChatCompletionMessage = run_llm(history, tools)
        add_messages([response])

        print(json.dumps(response.model_dump(), indent=2, default=str))

        if response.content:
            return get_messages()

        # Create a condition in case tool_calls is in response object
        if response.tool_calls:
            # Pull out the specific tool metadata from the response
            tool_call = response.tool_calls[0]

            # Run the tool locally
            tool_result = run_tool(tool_call, user_message)

            save_tool_response(tool_call.id, tool_result)

        max_turns+=1


In [4]:
user_message: str = input("Enter your question here:")
tools: list[ChatCompletionToolParam] = [{
    "type": "function",
    "function": {
        "name": "get_current_time", # <--- Your functions name
        "description": "Returns the current time as a string.", # <--- a description for the LLM
        "parameters": {}
    }
}]

run_agent(user_message, tools)

{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_FzQ5KyiXLADhTVGucgIMdxpJ",
      "function": {
        "arguments": "{}",
        "name": "get_current_time"
      },
      "type": "function"
    }
  ]
}
{
  "content": "The current time is 17:11:27.",
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": null
}


[{'role': 'user', 'content': 'What is the current time now?'},
 {'content': None,
  'refusal': None,
  'role': 'assistant',
  'annotations': [],
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_FzQ5KyiXLADhTVGucgIMdxpJ',
    'function': {'arguments': '{}', 'name': 'get_current_time'},
    'type': 'function'}]},
 {'role': 'tool',
  'tool_call_id': 'call_FzQ5KyiXLADhTVGucgIMdxpJ',
  'content': '17:11:27'},
 {'content': 'The current time is 17:11:27.',
  'refusal': None,
  'role': 'assistant',
  'annotations': [],
  'audio': None,
  'function_call': None,
  'tool_calls': None}]